# 01 — Streaming & Sketching: Critique Playbook + Runnable Verifiers
**This notebook covers ~40% of the exam** (P1a + all of P2): reading an LLM's answer about a streaming/sketching algorithm and stating what's right, what's wrong, and *why* — with correct parameter values.

**How to use in the exam.** For each critique question: (1) name the correct algorithm and its standard parameters from the reference tables below; (2) run the matching verifier cell to *empirically prove* the bug or confirm the claim; (3) write 3 right / 3 wrong (or the fix) citing the numbers the verifier printed. A critique backed by an executed demonstration ("I simulated it: it samples 100%, not 1% — see output") scores far above assertion.

> Pure Python + Spark built-ins only. `mmh3` may exist on Jojie; every cell here uses stdlib `hashlib` so it runs anywhere. Swap to `mmh3.hash(x, seed)` if you prefer — the logic is identical.

In [1]:
# ---- universal hash helper (hashlib-based; no external deps) ----
import hashlib, random, math
import numpy as np
random.seed(42); np.random.seed(42)

def h_int(x, seed=0):
    "Deterministic 64-bit-ish unsigned hash of any stringifiable x, salted by seed."
    b = f"{seed}:{x}".encode()
    return int(hashlib.md5(b).hexdigest(), 16)

print("helper ready: h_int(x, seed) ->", h_int("user_1", 0) % 1000)

helper ready: h_int(x, seed) -> 937


## P2a pattern — deterministic hash sampling (sample p% of keys, reproducible, no stored list)

**Correct method.** Hash the key to a fixed base B (e.g. 100 or a large power of two), keep if `hash(key) % B < p_count`. For 1%: `hash(id) % 100 < 1`. O(1) time, O(1) memory, reproducible (same id → same decision), no stored list.

**The classic planted bug** (past exam 2a): `bucket = hash % TARGET_PERCENTAGE` with `TARGET_PERCENTAGE=1` → `hash % 1 == 0` always → `if bucket < MODULUS_BASE(=100)` always true → **samples 100%, not 1%.** The two constants are swapped: percentage was used as the modulus, modulus as the threshold.

**Right (3):** hash-based rule is reproducible without storing ids; O(1)/O(1); explicit params given.
**Wrong (3):** `% 1` collapses to 0 (samples everything); role of the two constants swapped; no seed pinned for the hash (reproducibility depends on a fixed hash + seed).

In [2]:
# VERIFIER: run the buggy logic vs the correct logic on 100k ids, print actual sample rates.
N = 100_000
ids = [f"tracker_{i}" for i in range(N)]

def buggy_sample(tid, TARGET_PERCENTAGE=1, MODULUS_BASE=100, seed=42):
    bucket = h_int(tid, seed) % TARGET_PERCENTAGE     # % 1  -> always 0
    return bucket < MODULUS_BASE                        # 0 < 100 -> always True

def correct_sample(tid, p_percent=1, base=100, seed=42):
    return (h_int(tid, seed) % base) < p_percent       # 1-in-100

buggy_rate   = sum(buggy_sample(t)   for t in ids) / N
correct_rate = sum(correct_sample(t) for t in ids) / N
print(f"buggy   logic keeps {buggy_rate:.1%}  <-- claims 1%, actually samples everything")
print(f"correct logic keeps {correct_rate:.2%}  (target 1.00%)")
# reproducibility check: same ids -> identical decision set
again = sum(correct_sample(t) for t in ids) / N
print("reproducible:", correct_rate == again)

buggy   logic keeps 100.0%  <-- claims 1%, actually samples everything
correct logic keeps 1.02%  (target 1.00%)
reproducible: True


## P2b pattern — Bloom filter (membership, no false negatives, target FPR)

**Formulas (memorize).** For n items, target false-positive rate ε:
- bits: **m = ⌈ −n·ln ε / (ln 2)² ⌉** (round up to a convenient size)
- hashes: **k = round( (m/n)·ln 2 )**
- achieved FPR ≈ **(1 − e^(−kn/m))^k**

For n=500, ε=0.02: m ≈ −500·ln(0.02)/(ln2)² ≈ **4071 → use 4096**; k ≈ (4096/500)·ln2 ≈ 5.67 → **6**.

**Common bugs to catch/fix:** parameters unjustified or arbitrary (LLM's `M_BITS=2048, K=8` gives the wrong FPR); claiming "definitely present" (Bloom can only say *definitely absent* / *maybe present* — no false negatives, yes false positives); forgetting bits are shared so FPR rises with n.

In [3]:
# VERIFIER: build the Bloom filter with formula-chosen params, measure empirical FPR vs target.
def bloom_params(n, eps):
    m = math.ceil(-n * math.log(eps) / (math.log(2) ** 2))
    k = max(1, round((m / n) * math.log(2)))
    return m, k

n, eps = 500, 0.02
m, k = bloom_params(n, eps)
print(f"n={n}, target eps={eps}  ->  m={m} bits, k={k} hashes")

bits = [0] * m
approved = [f"dev_{i}" for i in range(n)]
for d in approved:
    for s in range(k):
        bits[h_int(d, s) % m] = 1

def maybe_present(x):
    return all(bits[h_int(x, s) % m] == 1 for s in range(k))

# no false negatives: every inserted item must return True
fn = sum(not maybe_present(d) for d in approved)
# empirical FPR on 50k never-inserted ids
trials = [f"other_{i}" for i in range(50_000)]
fp = sum(maybe_present(x) for x in trials) / len(trials)
theo = (1 - math.exp(-k * n / m)) ** k
print(f"false negatives: {fn} (must be 0)")
print(f"empirical FPR: {fp:.3%} | formula FPR: {theo:.3%} | target: {eps:.1%}")

n=500, target eps=0.02  ->  m=4072 bits, k=6 hashes


false negatives: 0 (must be 0)
empirical FPR: 2.168% | formula FPR: 2.008% | target: 2.0%


## P2c pattern — reservoir sampling (uniform size-k sample from a stream of unknown length)

**Algorithm R (Vitter).** Keep first k. For the i-th item (i>k, 1-indexed): draw j∈{1..i}; if j≤k, replace reservoir[j]. Each item seen so far ends with probability **k/N**. O(k) memory, one pass, N not needed in advance.

**What to check / fix:** index convention (draw j in 1..i, replace a slot 1..k — in 0-indexed code: `j=randint(0,i)`, replace if `j<k`); the equal-probability guarantee should be *justified* inductively (survival k/n · n/(n+1) = k/(n+1)); "running/window" wording — reservoir is all-time uniform, NOT a recent-window sample.

In [4]:
# VERIFIER: run reservoir sampling many times; every stream position should appear ~k/N of the time.
def reservoir(stream, k, rnd):
    res = []
    for i, x in enumerate(stream):
        if i < k:
            res.append(x)
        else:
            j = rnd.randint(0, i)      # 0..i inclusive
            if j < k:
                res[j] = x
    return res

N, k, TRIALS = 200, 20, 20_000
rnd = random.Random(0)
counts = np.zeros(N)
for _ in range(TRIALS):
    for x in reservoir(range(N), k, rnd):
        counts[x] += 1
emp = counts / TRIALS
print(f"target inclusion prob k/N = {k/N:.3f}")
print(f"empirical mean = {emp.mean():.3f} | min = {emp.min():.3f} | max = {emp.max():.3f}")
print("uniform across positions:", abs(emp.mean() - k/N) < 0.005 and emp.std() < 0.02)

target inclusion prob k/N = 0.100
empirical mean = 0.100 | min = 0.094 | max = 0.105
uniform across positions: True


## P2d pattern — HyperLogLog / Flajolet-Martin (count distinct, no stored set)

**Mechanics.** Hash each item; use first **p** bits to pick one of **m = 2^p** registers; register stores the max number of leading zeros (+1) of the rest. Estimate = **α_m · m² / Σ_j 2^(−M[j])**, with **α_m ≈ 0.7213/(1 + 1.079/m)**. **Standard error = 1.04/√m**; m=2^14 ⇒ <1% error. FM (single estimator) = 2^R/0.77351, high variance → HLL fixes it by stochastic averaging + harmonic mean.

**What LLM answers usually miss (the critique points):** no explicit params chosen (must state p / m / hash width); estimator is NOT the plain harmonic mean of registers — it uses α_m bias correction and the 2^(−M[j]) transform; small/large-range corrections omitted.

In [5]:
# VERIFIER: implement HLL, show measured error tracks 1.04/sqrt(m) across register counts.
def hll_estimate(items, p):
    m = 1 << p
    M = [0] * m
    for x in items:
        hv = h_int(x, 1)
        idx = hv & (m - 1)                 # first p bits
        w = hv >> p
        # position of lowest set bit (+1); leading-zeros-style rank on the remaining bits
        rank = 1
        while w and not (w & 1):
            rank += 1; w >>= 1
        M[idx] = max(M[idx], rank)
    alpha = 0.7213 / (1 + 1.079 / m)
    est = alpha * m * m / sum(2.0 ** (-v) for v in M)
    return est

true_n = 50_000
items = [f"class_{i}" for i in range(true_n)]
print(f"true distinct = {true_n}  (RMS error over 12 independent hash seeds vs theoretical SE)")
print(f"{'p':>3} {'m':>6} {'RMS rel.err':>12} {'1.04/sqrt(m)':>13}")
for p in (8, 10, 12, 14):
    errs = []
    for seed in range(12):                          # average many hashings: RMS error should ~ SE
        m = 1 << p; M = [0]*m
        for x in items:
            hv = h_int(x, 100 + seed); idx = hv & (m-1); w = hv >> p
            rank = 1
            while w and not (w & 1):
                rank += 1; w >>= 1
            M[idx] = max(M[idx], rank)
        alpha = 0.7213 / (1 + 1.079/m)
        est = alpha * m * m / sum(2.0**(-v) for v in M)
        errs.append((est - true_n) / true_n)
    rms = math.sqrt(sum(e*e for e in errs) / len(errs))
    print(f"{p:>3} {1<<p:>6} {rms:>11.2%} {1.04/math.sqrt(1<<p):>12.2%}   <- SE is a std-dev, matched by RMS")

true distinct = 50000  (RMS error over 12 independent hash seeds vs theoretical SE)
  p      m  RMS rel.err  1.04/sqrt(m)


  8    256       9.14%        6.50%   <- SE is a std-dev, matched by RMS


 10   1024       2.69%        3.25%   <- SE is a std-dev, matched by RMS


 12   4096       1.38%        1.62%   <- SE is a std-dev, matched by RMS


 14  16384       1.13%        0.81%   <- SE is a std-dev, matched by RMS


## P2e pattern — running IQR of a stream

**The interpretation fork (state it first).** "Running IQR" is ambiguous:
- **sliding window (size W):** exact IQR over the last W values — feasible with a FIFO queue + an ordered structure; **W is the required parameter** (e.g. W=300).
- **all-time:** exact IQR needs all values ⇒ unbounded memory. Use an **approximate quantile sketch** (GK / t-digest / `percentile_approx` in Spark, or a histogram).

**Bugs to catch:** parameter W left unspecified ("e.g. 300" is not a chosen value); claiming binary-search makes updates fast — search is O(log W) but insert/delete into a sorted list still **shifts O(W)** elements; quartile convention (interpolation, even-W handling) undefined.

In [6]:
# VERIFIER: sliding-window exact IQR (queue + bisect) and confirm insert cost is O(W), not O(log W).
import bisect
from collections import deque

def running_iqr_window(stream, W):
    q = deque(); srt = []
    out = []
    for x in stream:
        bisect.insort(srt, x); q.append(x)      # insort = O(W) shift despite O(log W) search
        if len(q) > W:
            old = q.popleft()
            del srt[bisect.bisect_left(srt, old)]
        if len(q) == W:
            q1 = srt[int(0.25 * (W - 1))]; q3 = srt[int(0.75 * (W - 1))]
            out.append(q3 - q1)
    return out

W = 300
rng = np.random.default_rng(0)
stream = rng.normal(72, 10, 5000)              # simulated heart-rate stream
iqrs = running_iqr_window(stream, W)
print(f"window W={W} | #IQR readings={len(iqrs)} | last IQR={iqrs[-1]:.2f}")
print(f"normal-dist theoretical IQR ~ 1.349*sigma = {1.349*10:.2f} (sanity check)")
# show the O(W) claim cleanly: time a fixed number of steady-state insert+delete ops as W grows
import time
def steady_state_update_cost(W, n_updates=4000):
    srt = sorted(rng.normal(72, 10, W).tolist())
    q = deque(srt)
    xs = rng.normal(72, 10, n_updates)
    t0 = time.perf_counter()
    for x in xs:
        bisect.insort(srt, x)                       # O(log W) search + O(W) shift
        old = q.popleft(); q.append(x)
        del srt[bisect.bisect_left(srt, old)]       # O(log W) search + O(W) shift
    return (time.perf_counter() - t0) / n_updates * 1e6   # microseconds per update
for Wt in (200, 800, 3200, 12800):
    print(f"  W={Wt:6d} -> {steady_state_update_cost(Wt):6.1f} us/update  "
          f"(rises ~linearly with W => per-update O(W), not O(log W))")

window W=300 | #IQR readings=4701 | last IQR=12.75
normal-dist theoretical IQR ~ 1.349*sigma = 13.49 (sanity check)
  W=   200 ->    0.7 us/update  (rises ~linearly with W => per-update O(W), not O(log W))
  W=   800 ->    0.9 us/update  (rises ~linearly with W => per-update O(W), not O(log W))
  W=  3200 ->    1.8 us/update  (rises ~linearly with W => per-update O(W), not O(log W))
  W= 12800 ->    5.6 us/update  (rises ~linearly with W => per-update O(W), not O(log W))


## Master reference — parameters & one-line "gotcha" for every streaming/sketching algorithm
| Algorithm | Purpose | Key params (standard values) | The gotcha graders test |
|---|---|---|---|
| Hash sampling | keep p% of keys | `hash%base < p_count`; base=100 for 1% | `%1`≡0 samples 100%; pin the seed |
| Bloom filter | membership | m=⌈−n·lnε/(ln2)²⌉, k=round((m/n)ln2) | **no false negatives**; can't say "definitely present" |
| Reservoir (Alg R) | uniform k-sample | k; p=k/N | all-time uniform, NOT a recent window |
| FM | count distinct | est=2^R/0.77351 | high variance alone → needs averaging |
| HyperLogLog | count distinct | m=2^p, α_m≈0.7213/(1+1.079/m), SE=1.04/√m; m=2^14⇒<1% | not plain harmonic mean; needs α_m + small/large-range fixes |
| Count-Min | point frequency | d=⌈ln(1/δ)⌉, w=⌈e/ε⌉, **min** query | **never undercounts**; overestimate ≤ ε·N |
| AMS | F₂ moment | est=N(2r−1), average M copies | unbiased; high F₂ = skew/anomaly |
| DGIM | 1s in window N | O(log²N) buckets, ≤2 per size | **halve the oldest bucket**; ≤50% error |
| Exp. decay | recency weight | weight (1−c)^age, keep w>½, window≈1/c | continuous, no hard boundary (unlike DGIM) |

**Critique answer skeleton (use every time):** name the correct algorithm → 3 things the LLM got right (with the correct formula) → 3 things wrong/missing (bug, unjustified param, or overstated guarantee) → the fix with explicit values → *run a verifier and cite its output.*